# Week 36: Data and Ruled-Based Baselines

In [ ]:
import nb_setup
import pandas as pd
from src.data import load_tokenised_data
from src.translator import NLLBTranslator
from src.utils import get_most_common_tokens

In [ ]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

In [ ]:
# 1. Number of examples, answerable/unanswerable proportions, median and interquantile range of
# tokenised question and context lengths (table sufficient, plots are optional), missing values, and
# exact duplicate question--context pairs. Everything by language and dataset split (train/validation).


def statistics_by_language(df, dataset_name):
    stats = df.groupby("lang").agg(
        num_examples=("lang", "size"),
        num_answerable=("answerable", "sum"),
        num_unanswerable=("answerable", lambda x: (~x).sum()),
        median_question_length=("question_tokens", lambda x: x.apply(len).median()),
        iqr_question_length=("question_tokens", lambda x: x.apply(len).quantile(0.75) - x.apply(len).quantile(0.25)),
        median_context_length=("context_tokens", lambda x: x.apply(len).median()),
        iqr_context_length=("context_tokens", lambda x: x.apply(len).quantile(0.75) - x.apply(len).quantile(0.25)),
        num_missing_values=("question", lambda x: x.isnull().sum() + df["context"].isnull().sum() + df["answer"].isnull().sum()),
        num_exact_duplicates=("question", lambda x: df.duplicated(subset=["question", "context"]).sum())
    ).reset_index()
    stats["answerable_proportion"] = (
        stats["num_answerable"] / stats["num_examples"]
    ).round(4)

    stats["unanswerable_proportion"] = (
        stats["num_unanswerable"] / stats["num_examples"]
    ).round(4)
    stats["dataset"] = dataset_name
    return stats


In [ ]:
stats_train = statistics_by_language(df_train_tokenised, "train")
stats_val = statistics_by_language(df_val_tokenised, "validation")

In [ ]:
stats_train

In [ ]:
stats_val

In [ ]:
# 2. The five most common question tokens in each language, along with their English translations.

translator = NLLBTranslator()

five_most_common_question_tokens = get_most_common_tokens(
    df_train_tokenised,
    translator,
    n=5
)

five_most_common_question_tokens

In [ ]:
# 3. Verify that every answerable item’s answer equals the substring beginning at answer start

def verify_answer_is_substring(df, dataset_name):
    answerable_df = df[df["answerable"]]
    failures = []
    num_checked = 0

    for _, row in answerable_df.iterrows():
        answer_start = row["answer_start"]
        answer_end = answer_start + len(row["answer"])
        if row["context"][answer_start:answer_end] != row["answer"]:
            failures.append(row)
        num_checked += 1

    return {
        "dataset": dataset_name,
        "num_checked": num_checked,
        "num_failures": len(failures),
        "failures": failures
    }


In [ ]:
train_verification = verify_answer_is_substring(df_train_tokenised, "train")
validation_verification = verify_answer_is_substring(df_val_tokenised, "validation")

verification_results = pd.DataFrame([train_verification, validation_verification])
verification_results